# 🤖 J-Quants Interactive Analyst

セッションベースの対話型分析環境です。
- `Session` = 1つの分析テーマ（スイング、マクロなど）
- 会話・計算結果は全てディスクに永続化されます
- カーネル再起動後も `Session("同じ名前")` で完全復元されます

In [1]:
import sys
from pathlib import Path

project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from dotenv import load_dotenv
load_dotenv(project_root / ".env")

# マジックコマンド %%chat を有効化
%load_ext src.agent.magic

## セッションの作成（または再開）

新規の場合は `data_range` を指定します。再開する場合は名前だけでOKです。

In [2]:
from src.agent.session import Session

# 新規セッション（データを読み込む）
session = Session(
    name="swing_20240515",
    data_range=("2024-05-15", "2024-05-15"),
    data_dir=str(project_root / "data"),
)

# 翌日の再開時はこれだけ:
# session = Session("swing_20240515", data_dir=str(project_root / "data"))

print(f"Session: {session.name}")
print(f"Data: {len(session.df)} rows")
print(f"Artifacts: {session.list_artifacts()}")

INFO:src.agent.core:Using model: gemini-2.5-flash
INFO:src.analyzer.runner:Loading 1 files...
INFO:src.agent.session:Session 'swing_20240515': Loaded 4359 records (2024-05-15 ~ 2024-05-15)


Session: swing_20240515
Data: 4359 rows
Artifacts: []


## AIとの対話

`%%chat` で日本語の指示を送るだけです。AIはデータの状態を常に把握しています。

In [3]:
%%chat
今日読み込んでいるデータの中で、終値が始値よりも5%以上高い銘柄（大きな陽線）を探して、出来高の多い順に並べてください。

*(**swing_20240515** が考え中...)*

INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
INFO:src.agent.session:Saved artifact: step_001.parquet (57 rows)


**🤖 swing_20240515**:

```python
# 終値が始値よりも5%以上高い銘柄をフィルタリング
condition = (df['AdjC'] / df['AdjO']) >= 1.05
output_df = df[condition]

# 出来高の多い順に並べ替え
output_df = output_df.sort_values(by='AdjVo', ascending=False)

# 結果の考察
if not output_df.empty:
    result = f"本日、終値が始値より5%以上高かった銘柄は {len(output_df)} 件ありました。\n" \
             f"特に出来高が多かった上位5銘柄は以下の通りです:\n" \
             f"{output_df[['Date', 'Code', 'AdjO', 'AdjC', 'AdjVo']].head().to_string(index=False)}"
else:
    result = "本日、終値が始値より5%以上高い銘柄は見つかりませんでした。"
```

---

✅ **実行完了**

> 本日、終値が始値より5%以上高かった銘柄は 57 件ありました。
特に出来高が多かった上位5銘柄は以下の通りです:
      Date  Code   AdjO   AdjC      AdjVo
2024-05-15 65740   12.0   13.8 84420000.0
2024-05-15 67400   18.0   19.0 22318800.0
2024-05-15 52160  252.0  328.0 16841600.0
2024-05-15 77710   95.0  103.0 11492400.0
2024-05-15 15180 1078.0 1136.0  8897500.0

,Date,Code,O,H,L,C,UL,LL,Vo,Va,AdjFactor,AdjO,AdjH,AdjL,AdjC,AdjVo
2740,2024-05-15,65740,1204.0,1384.0,1113.0,1384.0,1,0,844200.0,1.066995e+09,1.0,12.0,13.8,11.1,13.8,84420000.0
2823,2024-05-15,67400,18.0,19.0,18.0,19.0,0,0,22318800.0,4.036973e+08,1.0,18.0,19.0,18.0,19.0,22318800.0
2102,2024-05-15,52160,252.0,332.0,248.0,328.0,1,0,16841600.0,5.105079e+09,1.0,252.0,332.0,248.0,328.0,16841600.0
3405,2024-05-15,77710,95.0,107.0,92.0,103.0,1,0,11492400.0,1.183465e+09,1.0,95.0,107.0,92.0,103.0,11492400.0
128,2024-05-15,15180,5390.0,6210.0,5390.0,5680.0,0,0,1779500.0,1.020472e+10,1.0,1078.0,1242.0,1078.0,1136.0,8897500.0
1571,2024-05-15,41830,4455.0,4777.0,4389.0,4715.0,0,0,2765400.0,1.271678e+10,1.0,2227.5,2388.5,2194.5,2357.5,5530800.0
3996,2024-05-15,92270,1200.0,1320.0,1200.0,1290.0,0,0,2924300.0,3.738118e+09,1.0,1200.0,1320.0,1200.0,1290.0,2924300.0
1813,2024-05-15,45640,15.0,16.0,15.0,16.0,0,0,2861900.0,4.344950e+07,1.0,15.0,16.0,15.0,16.0,2861900.0
979,2024-05-15,31030,215.0,234.0,213.0,231.0,0,0,2635100.0,5.994758e+08,1.0,215.0,234.0,213.0,231.0,2635100.0
3714,2024-05-15,83460,338.0,370.0,338.0,359.0,0,0,2466500.0,8.795353e+08,1.0,338.0,370.0,338.0,359.0,2466500.0


## 深掘り（AIは前回の結果を知っている）

In [4]:
%%chat
その結果の中から、さらに終値が2000円以下の銘柄だけに絞り込んで。

*(**swing_20240515** が考え中...)*

INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
INFO:src.agent.session:Saved artifact: step_002.parquet (43 rows)


**🤖 swing_20240515**:

```python
# 前回の結果 (output_df) から、終値が2000円以下の銘柄をフィルタリング
condition = output_df['AdjC'] <= 2000
output_df = output_df[condition]

# 結果の考察
if not output_df.empty:
    result = f"終値が始値より5%以上高く、かつ終値が2000円以下だった銘柄は {len(output_df)} 件ありました。\n" \
             f"特に出来高が多かった上位5銘柄は以下の通りです:\n" \
             f"{output_df[['Date', 'Code', 'AdjO', 'AdjC', 'AdjVo']].head().to_string(index=False)}"
else:
    result = "終値が始値より5%以上高く、かつ終値が2000円以下の銘柄は見つかりませんでした。"
```

---

✅ **実行完了**

> 終値が始値より5%以上高く、かつ終値が2000円以下だった銘柄は 43 件ありました。
特に出来高が多かった上位5銘柄は以下の通りです:
      Date  Code   AdjO   AdjC      AdjVo
2024-05-15 65740   12.0   13.8 84420000.0
2024-05-15 67400   18.0   19.0 22318800.0
2024-05-15 52160  252.0  328.0 16841600.0
2024-05-15 77710   95.0  103.0 11492400.0
2024-05-15 15180 1078.0 1136.0  8897500.0

,Date,Code,O,H,L,C,UL,LL,Vo,Va,AdjFactor,AdjO,AdjH,AdjL,AdjC,AdjVo
2740,2024-05-15,65740,1204.0,1384.0,1113.0,1384.0,1,0,844200.0,1.066995e+09,1.0,12.0,13.8,11.1,13.8,84420000.0
2823,2024-05-15,67400,18.0,19.0,18.0,19.0,0,0,22318800.0,4.036973e+08,1.0,18.0,19.0,18.0,19.0,22318800.0
2102,2024-05-15,52160,252.0,332.0,248.0,328.0,1,0,16841600.0,5.105079e+09,1.0,252.0,332.0,248.0,328.0,16841600.0
3405,2024-05-15,77710,95.0,107.0,92.0,103.0,1,0,11492400.0,1.183465e+09,1.0,95.0,107.0,92.0,103.0,11492400.0
128,2024-05-15,15180,5390.0,6210.0,5390.0,5680.0,0,0,1779500.0,1.020472e+10,1.0,1078.0,1242.0,1078.0,1136.0,8897500.0
3996,2024-05-15,92270,1200.0,1320.0,1200.0,1290.0,0,0,2924300.0,3.738118e+09,1.0,1200.0,1320.0,1200.0,1290.0,2924300.0
1813,2024-05-15,45640,15.0,16.0,15.0,16.0,0,0,2861900.0,4.344950e+07,1.0,15.0,16.0,15.0,16.0,2861900.0
979,2024-05-15,31030,215.0,234.0,213.0,231.0,0,0,2635100.0,5.994758e+08,1.0,215.0,234.0,213.0,231.0,2635100.0
3714,2024-05-15,83460,338.0,370.0,338.0,359.0,0,0,2466500.0,8.795353e+08,1.0,338.0,370.0,338.0,359.0,2466500.0
689,2024-05-15,25850,5500.0,5900.0,5370.0,5790.0,0,0,510400.0,2.883771e+09,1.0,1375.0,1475.0,1342.5,1447.5,2041600.0


## 自分のコードで計算結果を操作する

In [5]:
# AIが計算した最新の output_df を自分で操作できる
if session.latest_output_df is not None:
    print(session.latest_output_df.describe())
else:
    print("まだ計算結果はありません。")

                      Date            O            H            L  \
count                   43    43.000000    43.000000    43.000000   
mean   2024-05-15 00:00:00  1181.697674  1299.453488  1163.269767   
min    2024-05-15 00:00:00    15.000000    16.000000    15.000000   
25%    2024-05-15 00:00:00   323.000000   354.500000   322.500000   
50%    2024-05-15 00:00:00   926.000000  1054.000000   926.000000   
75%    2024-05-15 00:00:00  1427.000000  1601.500000  1363.500000   
max    2024-05-15 00:00:00  5500.000000  6210.000000  5390.000000   
std                    NaN  1288.311825  1418.451532  1272.739234   

                 C            Vo            Va  AdjFactor         AdjO  \
count    43.000000  4.300000e+01  4.300000e+01       43.0    43.000000   
mean   1270.209302  1.885377e+06  9.303498e+08        1.0   770.565116   
min      16.000000  2.400000e+03  1.160000e+06        1.0    12.000000   
25%     345.000000  1.347500e+05  4.415975e+07        1.0   338.550000   
50%    1

## 保存された計算結果（artifacts）の確認

In [ ]:
# 各ステップの結果はParquetとして永続化されている
print("Saved artifacts:", session.list_artifacts())

# 過去のステップの結果を呼び出すこともできる
# df_step1 = session.load_artifact(1)